# Re-junte 2027 · El escritorio del comité
Una invitación personalizada para Compadres en Tarija. Este cuaderno recupera el proyecto, edita datos, construye el sitio estático y exporta dos ZIP. La interfaz vive en `src/`; el cuaderno no contiene copias de HTML, CSS ni JavaScript.

**Uso:** en Colab, selecciona Python 3 y hardware «Ninguno». Ejecuta de arriba abajo. Puedes editar el evento, invitados y opciones visuales. Los nombres son ficticios; los datos publicados serán públicos. La aceptación del destino no registra asistencia.

GitHub conserva el código. `/content/` es temporal: exporta y sube tus cambios antes de cerrar la sesión. No se requieren Node.js, npm, instalaciones locales, GPU ni tokens de GitHub.

## 1. Preparar y recuperar
Se trabajará en `/content/re-junte/`. Elige `github` o `zip`. En una sesión nueva se clona la rama `main` del repositorio público. Para un repositorio privado usa un ZIP descargado desde GitHub; no pegues credenciales.

Si la carpeta ya existe se reutiliza: no se hace `pull`, no se sobrescribe y no se borra tu trabajo. Para recuperar otra versión, exporta los cambios y abre un entorno de ejecución nuevo. El ZIP **FUENTE** incluye el cuaderno y scripts; el ZIP **SITIO** no sirve para recuperar fuentes.

In [ ]:
from pathlib import Path, PurePosixPath
import csv, io, json, shutil, stat, subprocess, sys, tempfile, zipfile

ROOT = Path('/content/re-junte')
MODO = 'github'  # 'github' o 'zip'
REPOSITORIO = 'https://github.com/X3rtr3B/re-junte.git'
RAMA = 'main'

if ROOT.exists():
    if not (ROOT / 'scripts/build.py').is_file():
        raise RuntimeError('La carpeta existente no es un proyecto completo. Expórtala y usa una sesión nueva; no se sobrescribirá.')
    print('Proyecto existente conservado:', ROOT)
else:
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=ROOT.parent) as temporary:
        staging = Path(temporary) / 'recuperado'
        if MODO == 'github':
            subprocess.run(['git', 'clone', '--depth', '1', '--branch', RAMA, REPOSITORIO, str(staging)], check=True)
        elif MODO == 'zip':
            from google.colab import files
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError('Carga exactamente un ZIP FUENTE.')
            archive_bytes = next(iter(uploaded.values()))
            extracted = Path(temporary) / 'zip'
            with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                if len(archive.infolist()) > 10000 or sum(info.file_size for info in archive.infolist()) > 512 * 1024 * 1024:
                    raise ValueError('ZIP demasiado grande (máximo 512 MiB descomprimidos).')
                for info in archive.infolist():
                    path = PurePosixPath(info.filename)
                    mode = info.external_attr >> 16
                    if path.is_absolute() or '..' in path.parts or '\\' in info.filename or ':' in info.filename or stat.S_ISLNK(mode):
                        raise ValueError('ZIP con una ruta o enlace no permitido.')
                archive.extractall(extracted)
            candidates = [p.parent for p in extracted.rglob('ReJunte.ipynb') if (p.parent / 'scripts/build.py').is_file()]
            if len(candidates) != 1:
                raise ValueError('No se encontró un único proyecto fuente completo.')
            shutil.copytree(candidates[0], staging)
        else:
            raise ValueError('MODO debe ser github o zip.')
        if not (staging / 'scripts/build.py').exists():
            raise ValueError('La versión recuperada no contiene la aplicación. Comprueba la rama.')
        shutil.move(str(staging), ROOT)

sys.path.insert(0, str(ROOT / 'scripts')) if str(ROOT / 'scripts') not in sys.path else None
print('Python:', sys.version.split()[0], '| Proyecto:', ROOT)

## 2. Consultar y editar el evento
La URL pública permanece vacía hasta que compruebes una dirección publicada real. Cambia valores en `CAMBIOS` y vuelve a ejecutar. Los campos no mencionados se conservan.

`texto` contiene los párrafos configurables. El texto aprobado se entrega íntegro: no cambies estos párrafos salvo que decidas aprobar una nueva versión. Las cláusulas y respuestas de los botones se editan en `src/index.html` y `src/app.js`. Fecha, ciudad, festividad y concordancia se aplican automáticamente.

In [ ]:
from common import load, validate_data

evento, invitados = load(ROOT)
print(json.dumps(evento, ensure_ascii=False, indent=2))

CAMBIOS = {
    # 'url_publica': 'PEGA_AQUI_LA_DIRECCION_REAL_DESPUES_DE_PUBLICAR',
    # 'video_habilitado': False,
    # 'opciones_visuales': {'animaciones': True, 'confeti': True},
}
if CAMBIOS:
    evento.update(CAMBIOS)
    validate_data(evento, invitados, allow_missing_ids=True)
    (ROOT / 'config/evento.json').write_text(json.dumps(evento, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Configuración guardada.')

## 3. Administrar invitados
El archivo principal es `config/invitados.csv` (UTF-8). Puedes abrirlo en el panel Archivos de Colab con doble clic. Conserva exactamente la cabecera `id,nombre,apodo,articulo_alias,forma_obligacion`. Usa comillas CSV si un nombre incluye comas. Deja el ID vacío solo para personas nuevas.

La siguiente celda consulta la lista y ofrece una edición por ID que conserva el enlace. Descomenta una llamada para agregar o modificar. **No deducimos género:** elige `obligado` u `obligada` y un artículo (`el`, `la`, `los`, `las` o vacío). El apodo va sin artículo cuando lo configuras aparte. Se rechazan duplicados exactos de nombre y apodo; personas distintas pueden compartir nombre.

In [ ]:
from guests import save_guest

evento, invitados = load(ROOT)
for persona in invitados:
    print(persona)

# Agregar (ejemplo; no se agrega nada por defecto):
# save_guest('Valeria', 'Vale', '', 'obligada', root=ROOT)

# Modificar: copia el ID real impreso arriba; se mantiene sin cambios.
# save_guest('Camila', 'Macha', 'la', 'obligada', guest_id='COPIA_EL_ID_EXISTENTE', root=ROOT)

# Vuelve a consultar después de guardar:
# print((ROOT / 'config/invitados.csv').read_text(encoding='utf-8'))

## 4. Asignar IDs y construir
`build.py` valida primero, usa `secrets.token_urlsafe(18)` solo para filas sin ID y guarda la correspondencia en el CSV. Copia `src/` a `docs/`, genera `data.js` y `.nojekyll`. Volver a ejecutarlo mantiene todos los IDs existentes.

**Importante:** exporta el ZIP FUENTE y sube el CSV actualizado a GitHub. Si pierdes una sesión antes de conservar los IDs nuevos, también pierdes su correspondencia. No repartas enlaces antes de guardar el CSV.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/build.py')], cwd=ROOT, check=True)

## 5. Validar
Comprueba datos, unicidad, sincronización, UTF-8 y recursos. Las pruebas Python no requieren dependencias. Las pruebas reales de navegador se ejecutaron en Codex y tienen instrucciones propias; Playwright no es obligatorio en Colab.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/validate.py')], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_*.py', '-v'], cwd=ROOT, check=True)

## 6. Vista previa temporal
El servidor sirve únicamente `docs/`. Al repetir se cierra el servidor anterior y se selecciona un puerto libre. La vista previa vive solo durante la sesión: **no es el enlace público para los invitados**.

Si el iframe bloquea navegación, video o foco, usa la ventana ofrecida por Colab o prueba en GitHub Pages. Allí verifica también Atrás/Adelante y los enlaces `?i=ID`. No necesitas ejecutar Colab para que la web publicada funcione.

In [ ]:
import preview
puerto = preview.start(ROOT)
print(f'Servidor temporal en puerto {puerto}; sirve docs/.')
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(puerto, height=850)
except ImportError:
    print(f'Fuera de Colab: abre http://127.0.0.1:{puerto}/ en esta misma máquina.')
# Para detener: preview.stop()

## 7. Exportar y descargar
Se generan `re-junte-FUENTE.zip` (proyecto recuperable) y `re-junte-SITIO-PAGES.zip` (solo web, con `index.html` en la raíz). Si la URL pública está configurada se genera además `enlaces_personalizados.csv` con nombres Unicode.

**Extrae los ZIP antes de subir a GitHub.** Subir un archivo ZIP al repositorio no publica su contenido. El CSV de enlaces no se incluye en `docs/`. Si una nueva ejecución no ofrece descargas múltiples, descarga cada archivo desde el panel Archivos.

In [ ]:
from importlib import import_module
exportar = import_module('export').export
fuente, sitio, enlaces = exportar(ROOT)
try:
    from google.colab import files
    for archivo in (fuente, sitio, enlaces):
        if archivo:
            files.download(str(archivo))
except ImportError:
    print('Archivos disponibles en', ROOT / 'exports')

## 8. Publicar en GitHub Pages
1. Abre tu repositorio en GitHub. Para la primera publicación, extrae **FUENTE** y sube su contenido (el cuaderno, `src/`, `config/`, scripts y `docs/`) a `main`, usando **Add file → Upload files**. Conserva `.nojekyll`; si el explorador oculta archivos con punto, compruébalo en GitHub. No subas `exports/` ni credenciales.
2. En **Settings → Pages**, elige **Source: Deploy from a branch**, **Branch: main**, **Folder: /docs** y guarda.
3. Espera a que GitHub muestre la dirección y ábrela. No presupongas que ya está publicada: comprueba selector y un enlace individual.
4. Pega esa dirección real en `CAMBIOS['url_publica']`, ejecuta configuración, construcción, validación y exportación. Sube los archivos actualizados; ya puedes distribuir el CSV de enlaces.
5. Para proyectos de usuario `USUARIO.github.io`, la ruta puede ser `/`; para repositorios de proyecto normalmente incluye `/re-junte/`. Siempre usa la dirección que muestra GitHub.

Las vistas previas de mensajería son genéricas, dependen del servicio y no se personalizan por invitado. No se hace `git push` automático ni se solicitan tokens.

## 9. Persistencia opcional en Drive
Las funciones básicas funcionan sin Drive. Para una copia adicional activa `USAR_DRIVE` abajo: Colab te solicitará acceso a Drive como parte de su montaje. Se copian únicamente los ZIP exportados a una carpeta identificada. No se usa Drive como fuente implícita ni como alojamiento del sitio.

In [ ]:
USAR_DRIVE = False
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    destino = Path('/content/drive/MyDrive/ReJunte-respaldos')
    destino.mkdir(parents=True, exist_ok=True)
    from datetime import datetime, timezone
    version = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    for archivo in (fuente, sitio, enlaces):
        if archivo:
            shutil.copy2(archivo, destino / f'{version}-{archivo.name}')
    print('Respaldo guardado:', destino)
else:
    print('Drive desactivado; conserva las descargas y tu versión de GitHub.')

## 10. Mantenimiento sin perder enlaces
- Antes de editar, recupera la última versión de `main`. No trabajes a la vez sobre copias antiguas en varias sesiones.
- Modifica invitados en el CSV; conserva los IDs. Editar nombre, apodo, artículo o forma no requiere cambiarlos. No reutilices el ID de una persona eliminada.
- Sustituye recursos en `src/assets/`; para video añade MP4 y subtítulos VTT reales y activa la opción. Nunca hay reproducción automática ni música por defecto.
- Cambia diseño en `src/`; `docs/` se regenera. No edites solo `docs/`, porque la siguiente construcción sustituirá esos cambios.
- Reconstruye, valida, previsualiza y exporta. Descarga **FUENTE**, conserva el CSV y sube los cambios a GitHub. Compara la última versión remota antes de sobrescribir archivos desde el navegador.
- Publica nuevamente desde `main` y `/docs`; comprueba un enlace antiguo. Los enlaces dependen de conservar tanto el ID como la URL base del sitio.
- El hosting puede mantener sus propios registros técnicos; este proyecto no incorpora analítica, cookies de seguimiento ni un servidor de confirmación.

El código no se duplica dentro del cuaderno. Cerrar Colab no afecta a la aplicación ya publicada en Pages.